# View Purge Behavior — full gate matrix

> **New here? Read this first.** A view drop's outcome depends on FOUR things, and
> this notebook maps how they interact — including isolating which one causes the
> `403` that an earlier session saw.

## The four dimensions

```
purge-view-metadata-on-drop   catalog config — should the drop delete the metadata file?
drop-with-purge.enabled       catalog config — is purge allowed at all?
?purgeRequested=              request param  — true / false / absent
CATALOG_MANAGE_CONTENT        privilege      — does the principal have purge rights?
```

In our no-STS environment, even a successful purging drop **orphans** the metadata
file (the async cleanup task fails — Apache Polaris #379, see the test report).

## The 8-case matrix

```
C1  meta=true  purge=false  grant=NO   req=absent   ← the original "403" scenario
C2  meta=true  purge=false  grant=YES  req=absent   isolates the PRIVILEGE gate
C3  meta=true  purge=true   grant=NO   req=absent   isolates the CONFIG gate
C4  meta=true  purge=true   grant=YES  req=absent   everything enabled
C5  meta=true  purge=true   grant=YES  req=false    explicit purgeRequested=false
C6  meta=false purge=true   grant=YES  req=true     param vs config conflict
C7  meta=false purge=true   grant=YES  req=absent   metadata-only baseline
C8  drop an already-dropped view                    404 idempotency
```

The final cell derives, from the observed status codes, **which gate produces the
403** (privilege vs config) — answering the open question rather than assuming it.

## Setup note (important)

```
A grant can't be cleanly REVOKED mid-test, so the runner uses TWO catalogs:
  view-test-grant     — CATALOG_MANAGE_CONTENT granted
  view-test-nogrant   — NOT granted
Each case runs against the right one. Both are cleaned up (verified) at the end.
```

## Prerequisites & how to run

```
Files:   polaris_test_utils.py, minio_rest.py, config/  (same folder)
Packages: pip install requests pyyaml opensearch-py
Run:     setup cell (🟢/🔴 banner) → force_delete_catalog def → matrix setup →
         C1–C8 → truth table → cleanup.
Switch:  change init_env("dev") → init_env("prod") in setup, re-run that cell.

⚠️  Storage via MinIO REST API (minio_rest.py) — NO s3fs. No hardcoded creds.
```

---


In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

# ── Setup ─────────────────────────────────────────────────────────────────
# Environment switch HERE. Default 🟢 LOCAL. For shared clusters: init_env("dev")/"prod"
# and re-run THIS cell to switch (no kernel restart needed). "prod" = availability only.
from polaris_test_utils import *
init_env("local")
require_not_prod("purge behavior test")  # PROD-safe: refuses to run against company PROD

import uuid, time
tok = root_token()

# Map friendly kwarg names → the EXACT Polaris property keys.
# (Property names mix hyphens AND dots — a blind _→- replace produces the wrong
#  key, e.g. "drop-with-purge-enabled" instead of "drop-with-purge.enabled",
#  which Polaris rejects with 400. That silent failure caused earlier wrong results.)
_CONFIG_KEYS = {
    "drop_with_purge_enabled":       "polaris.config.drop-with-purge.enabled",
    "purge_view_metadata_on_drop":   "polaris.config.purge-view-metadata-on-drop",
}

def set_catalog_config(catalog, _retries=5, **flags):
    """Merge config flags into a catalog AND VERIFY they applied.
    Uses the exact property keys, checks the PUT response, retries on 409
    entityVersion conflict, and confirms the values changed by reading back."""
    want = {}
    for k, v in flags.items():
        if k not in _CONFIG_KEYS:
            raise KeyError(f"unknown config flag '{k}'. Known: {list(_CONFIG_KEYS)}")
        want[_CONFIG_KEYS[k]] = v
    for attempt in range(_retries):
        cat = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json()
        merged = {**cat.get("properties", {}), **want}
        r = requests.put(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok),
                         json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
        if r.status_code in (200, 204):
            now = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json().get("properties", {})
            if all(now.get(k) == v for k, v in want.items()):
                return True
        elif r.status_code == 409:
            time.sleep(0.2); continue
        else:
            raise RuntimeError(f"set_catalog_config PUT failed [{r.status_code}]: {r.text[:200]}")
    now = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json().get("properties", {})
    if not all(now.get(k) == v for k, v in want.items()):
        raise RuntimeError(f"set_catalog_config could not apply {want} to {catalog} "
                           f"(now: { {k: now.get(k) for k in want} })")
    return True

# Debug verbosity: False = compact per-case; True = full trace every case.
# 4xx/5xx ALWAYS escalate to full trace regardless of this flag.
DEBUG = False

print("✅ setup ready (correct property keys + verified writes)")


🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ setup ready (correct property keys + verified writes)


In [2]:
# HARDENED cleanup — removes a catalog completely and VERIFIES it's gone.
# Key fixes:
#  • drop views/tables WITHOUT ?purgeRequested (avoids the purge-coupling that
#    can make a view drop fail and leave the namespace non-empty)
#  • set purge-view-metadata=false BEFORE dropping views (so drop is metadata-only)
#  • verify namespace is empty before deleting it; verify catalog gone at the end
# A stale catalog with old config flags is what corrupted earlier test runs.

def _ns_name(ns):
    return ns[0] if isinstance(ns, list) else ns

def force_delete_catalog(cat):
    if requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code == 404:
        return True
    # make teardown safe: purge OFF for views (metadata-only drop), purge ON allowed
    set_catalog_config(cat, drop_with_purge_enabled="true", purge_view_metadata_on_drop="false")
    grant_privilege(catalog=cat, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)

    nss = requests.get(f"{BASE_CAT}/{cat}/namespaces", headers=h(tok)).json().get("namespaces", [])
    # deepest namespaces first
    nss = sorted(nss, key=lambda n: len(n if isinstance(n, list) else [n]), reverse=True)
    for ns in nss:
        nsn = _ns_name(ns)
        # drop all tables (plain drop — no purge; storage swept separately)
        for t in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", []):
            rt = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables/{t['name']}", headers=h(tok))
            if rt.status_code not in (200,204):
                print(f"   ⚠️ table {t['name']} drop: {rt.status_code} {rt.text[:80]}")
        # drop all views (plain drop — no purge)
        for v in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
            rv = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views/{v['name']}", headers=h(tok))
            if rv.status_code not in (200,204):
                print(f"   ⚠️ view {v['name']} drop: {rv.status_code} {rv.text[:80]}")
        # confirm empty, then delete namespace
        tleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", [])
        vleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", [])
        if tleft or vleft:
            print(f"   ❌ namespace {nsn} still has tables={len(tleft)} views={len(vleft)} — cannot delete")
            continue
        rns = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}", headers=h(tok))
        if rns.status_code not in (200,204):
            print(f"   ⚠️ namespace {nsn} delete: {rns.status_code} {rns.text[:100]}")

    # drop non-default catalog-roles (a catalog with extra roles is "not empty")
    for cr in requests.get(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles", headers=h(tok)).json().get("roles", []):
        crn = cr["name"] if isinstance(cr, dict) else cr
        if crn != "catalog_admin":
            rcr = requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{crn}", headers=h(tok))
            if rcr.status_code not in (200,204):
                print(f"   ⚠️ catalog-role {crn} delete: {rcr.status_code} {rcr.text[:80]}")
    rc = requests.delete(f"{BASE_MGMT}/catalogs/{cat}?purgeRequested=true", headers=h(tok))
    mc.delete_prefix(f"{cat}/", min_size=0)
    still = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code != 404
    if still:
        print(f"   ❌ catalog {cat} STILL EXISTS (delete status {rc.status_code}): {rc.text[:150]}")
        # show what's blocking
        leftover = requests.get(f"{BASE_CAT}/{cat}/namespaces", headers=h(tok)).json().get("namespaces", [])
        print(f"      remaining namespaces: {leftover}")
        return False
    return True

print("✅ force_delete_catalog ready")


✅ force_delete_catalog ready


## Test matrix — 8 cases across config × privilege × purgeRequested

We map which gate controls a view drop. Four dimensions interact:

```
purge-view-metadata-on-drop   catalog config (true/false)
drop-with-purge.enabled       catalog config (true/false)
?purgeRequested=              request param  (true/false/absent)
CATALOG_MANAGE_CONTENT        principal privilege (granted/not)
```

Because a grant can't be cleanly revoked mid-test, the runner uses **two catalogs**:
one WITH the grant and one WITHOUT. Each case runs against the right one.

```
C1  purge-meta=true  drop-purge=false  grant=NO   purgeReq=absent   ← original 403?
C2  purge-meta=true  drop-purge=false  grant=YES  purgeReq=absent   isolates privilege
C3  purge-meta=true  drop-purge=true   grant=NO   purgeReq=absent   isolates config
C4  purge-meta=true  drop-purge=true   grant=YES  purgeReq=absent   everything on
C5  purge-meta=true  drop-purge=true   grant=YES  purgeReq=false    explicit false
C6  purge-meta=false drop-purge=true   grant=YES  purgeReq=true     param vs config
C7  purge-meta=false drop-purge=true   grant=YES  purgeReq=absent   metadata-only baseline
C8  (drop an already-dropped view)                                  404 idempotency
```


In [3]:
# Per-case catalogs: each case creates its OWN dedicated catalog with the config
# baked in at create time and VERIFIED (see fresh_catalog_for_case). This replaces
# the earlier "two shared catalogs + flip properties" design, which was unreliable
# because property flips could silently fail. NS name used by all cases:
# (namespace name is provided by _ns() — unique per run)
print("✅ using per-case dedicated catalogs (config verified at creation)")


✅ using per-case dedicated catalogs (config verified at creation)


### Verify privileges BEFORE running the matrix

Confirms the actual identity + grants we're testing with, so the results are
interpretable. Specifically: which principal the token belongs to, and what
grants exist on each catalog's `catalog_admin` role. If the token is the **root
principal**, it may bypass privilege checks — which would explain why a
privilege-gated 403 doesn't appear for any matrix case.


In [4]:
# Verify WHO we authenticate as (privilege context for the whole test).
import base64, json as _json
def token_principal(t):
    try:
        p = t.split(".")[1]; p += "="*(-len(p)%4)
        c = _json.loads(base64.urlsafe_b64decode(p))
        return c.get("sub") or c.get("principal") or c.get("client_id")
    except Exception as e:
        return f"(decode failed: {e})"
print("Token principal:", token_principal(tok), "| ROOT_CLIENT:", ROOT_CLIENT)
pr = requests.get(f"{BASE_MGMT}/principal-roles", headers=h(tok))
if pr.status_code==200:
    print("Principal-roles:", [r["name"] for r in pr.json().get("roles", [])])
print("→ root maps to service_admin (bypasses catalog-role privilege checks),")
print("  which is why C9 uses a separate NON-root principal to test privilege.")


Token principal: root | ROOT_CLIENT: root
→ root maps to service_admin (bypasses catalog-role privilege checks),
  which is why C9 uses a separate NON-root principal to test privilege.


In [5]:
# Settle delay (seconds) after a read-after-write 500, to let a lagging PG-HA
# replica catch up before the next list/drop read. Pure infra mitigation; the
# real fix is routing Polaris metastore reads to the primary.
_RAW_SETTLE = 0.6

results = {}
import re as _re, json as _json

# CRITICAL: namespace name must be UNIQUE PER CATALOG (not per run).
# Polaris resolves namespaces BY NAME via a per-realm passthrough resolver. If the
# SAME namespace name exists across sibling catalogs in the realm, resolution
# returns an empty/ambiguous path (resolvedPath:[]) → size mismatch → null → NPE
# → 500 on namespace AND view CREATE. Proven by logs: 2 catalogs sharing a name
# mostly works; 4 catalogs sharing a name → 500 every time. Deriving the namespace
# from the catalog name guarantees uniqueness and avoids the collision.
def _ns_for(cat):
    # short, stable, unique-per-catalog namespace name
    return f"ns-{cat}"
# Back-compat shim: code paths that call _ns() without a catalog use a per-run
# fallback, but ALL create/use paths below pass the catalog explicitly.
_NS_FALLBACK = f"vp-ns-{uuid.uuid4().hex[:8]}"
def _ns(cat=None): return _ns_for(cat) if cat else _NS_FALLBACK

def _show(label, obj):
    """Pretty-print a dict/json blob under a label, indented."""
    try:
        txt = _json.dumps(obj, indent=2) if isinstance(obj, (dict, list)) else str(obj)
    except Exception:
        txt = str(obj)
    for ln in txt.splitlines():
        print(f"      {ln}")

def get_catalog_config(cat):
    props = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).json().get("properties", {})
    return {"purge-view-metadata-on-drop": props.get("polaris.config.purge-view-metadata-on-drop"),
            "drop-with-purge.enabled":      props.get("polaris.config.drop-with-purge.enabled")}

def make_view(cat, name):
    """Create a view and VERIFY it exists (raise on failure)."""
    body = {"name": name, "default-namespace": [_ns(cat)],
            "schema":{"type":"struct","schema-id":0,
                      "fields":[{"id":1,"name":"id","type":"long","required":True}]},
            "view-version":{"version-id":1,"timestamp-ms":int(time.time()*1000),"schema-id":0,
                            "default-namespace":[_ns(cat)],"summary":{"engine":"spark"},
                            "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
            "location": f"s3a://{BUCKET}/{cat}/{_ns(cat)}/{name}/"}
    url = f"{BASE_CAT}/{cat}/namespaces/{_ns(cat)}/views"
    # The view-create 500 (NullPointerException: "metadata") is an INTERMITTENT
    # read-after-write failure (PG-Pool/PG-HA replica lag): Polaris COMMITS the
    # view to the DB primary, then immediately re-resolves it to build the
    # response; if that read hits a lagging replica it returns an empty path →
    # NPE → 500. The commit SUCCEEDED — the 500 is a FALSE failure.
    #
    # TRUST-THE-COMMIT: a 200/201 is success; a 500 with body "metadata" /
    # NullPointerException is ALSO treated as a logical success (the entity is
    # written). We do NOT block on a verify-read, because that read suffers the
    # SAME lag and is unreliable. A short settle delay lets later list/drop
    # reads catch up. Any OTHER status (4xx, etc.) is a real failure and raised.
    r = requests.post(url, headers=h(tok), json=body)
    if r.status_code in (200, 201):
        return r, url, body
    if r.status_code == 500 and ("metadata" in r.text or "NullPointer" in r.text):
        time.sleep(_RAW_SETTLE)   # let the replica catch up before later reads
        class _OK:
            status_code = 200
            text = '{"note":"view committed; 500 was a read-after-write lag (false failure)"}'
        return _OK(), url, body
    return r, url, body   # genuine error — caller decides

def cat_files(cat, name):
    return mc.list(f"{cat}/{_ns(cat)}/{name}/")

def create_namespace_verified(cat, ns_name):
    """Create a namespace, tolerating the intermittent read-after-write 500
    (PG-Pool/PG-HA replica lag). TRUST-THE-COMMIT: a 500 with body 'metadata' /
    NullPointerException means the namespace WAS committed (false failure); we
    accept it and add a short settle delay rather than blocking on a verify-read
    that suffers the same lag."""
    r = create_namespace(catalog=cat, ns=ns_name, token=tok)
    if r.status_code in (200, 201):
        return True
    if r.status_code == 500 and ("metadata" in r.text or "NullPointer" in r.text):
        time.sleep(_RAW_SETTLE)   # committed; let the replica catch up
        return True
    raise RuntimeError(f"namespace {ns_name} create failed [{r.status_code}]: {r.text[:160]}")

def fresh_catalog_for_case(case, purge_meta, drop_purge, grant):
    cat = f"vptest-{case.lower()}"
    force_delete_catalog(cat)
    create_catalog(cat, token=tok, properties={
        "polaris.config.purge-view-metadata-on-drop": str(purge_meta).lower(),
        "polaris.config.drop-with-purge.enabled":     str(drop_purge).lower()})
    cfg = get_catalog_config(cat)
    assert cfg["purge-view-metadata-on-drop"] == str(purge_meta).lower(), f"{cat}: {cfg}"
    assert cfg["drop-with-purge.enabled"] == str(drop_purge).lower(), f"{cat}: {cfg}"
    if grant:
        grant_privilege(catalog=cat, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)
    create_namespace_verified(cat, _ns(cat))
    return cat, cfg

def run_case(case, purge_meta, drop_purge, grant, purge_requested, expected_status=(200,204),
             create_with_purge_enabled=True):
    """One case with rich tracing.
       DEBUG=True → full trace always. UNEXPECTED 4xx/5xx → full trace auto.
       Expected 4xx/5xx (in expected_status) → compact line only.
       create_with_purge_enabled: WORKAROUND for the view-create NPE — create the
       view while drop-with-purge=true, then set the intended drop_purge before
       the drop. Set False to create directly under the target config."""
    if isinstance(expected_status, int): expected_status = (expected_status,)

    # Create the catalog. If working around the create-NPE and the target is
    # drop_purge=false, create under TRUE first, then flip after the view exists.
    create_purge = True if (create_with_purge_enabled and not drop_purge) else drop_purge
    cat, cfg_before = fresh_catalog_for_case(case, purge_meta, create_purge, grant)
    vname = f"{case}-{uuid.uuid4().hex[:6]}"
    cr, cv_url, cv_body = make_view(cat, vname)

    # If CREATE failed, record it as a create-error and STOP this case (don't crash run)
    if cr.status_code not in (200, 201):
        results[case] = {"cat":cat, "vname":vname, "purge_meta":purge_meta, "drop_purge":drop_purge,
                         "grant":grant, "purge_req":{True:"true",False:"false",None:"absent"}[purge_requested],
                         "status":"CREATE_FAIL", "create_status":cr.status_code, "gone":None,
                         "before":0, "after":0,
                         "cfg_verified":(cfg_before["purge-view-metadata-on-drop"], cfg_before["drop-with-purge.enabled"]),
                         "err":cr.text[:200], "expected":expected_status}
        print(f"[{case}] ❌ VIEW CREATE FAILED [{cr.status_code}] on cat={cat} "
              f"(cfg: purge-meta={cfg_before['purge-view-metadata-on-drop']}, "
              f"drop-purge={cfg_before['drop-with-purge.enabled']})")
        print(f"   ── create failure trace ──")
        print(f"   POST {cv_url}")
        print(f"   request body:"); _show("", cv_body)
        print(f"   config: "); _show("", cfg_before)
        print(f"   response [{cr.status_code}]:"); _show("", cr.text)
        return results[case]

    # Apply the INTENDED drop-with-purge config now that the view exists
    # (this is the workaround: create under purge=true, test-drop under the target).
    if create_purge != drop_purge:
        set_catalog_config(cat, drop_with_purge_enabled=str(drop_purge).lower())
    cfg_before = get_catalog_config(cat)   # re-read: this is the config AT DROP TIME

    # STATE BEFORE
    files_before = cat_files(cat, vname)
    view_exists_before = requests.get(
        f"{BASE_CAT}/{cat}/namespaces/{_ns(cat)}/views/{vname}", headers=h(tok)).status_code == 200

    # BUILD DROP REQUEST
    url = f"{BASE_CAT}/{cat}/namespaces/{_ns(cat)}/views/{vname}"
    if purge_requested is True:    url += "?purgeRequested=true"
    elif purge_requested is False: url += "?purgeRequested=false"
    drop_t0 = time.time()
    r = requests.delete(url, headers=h(tok))
    elapsed = time.time() - drop_t0

    # STATE AFTER
    view_exists_after = requests.get(
        f"{BASE_CAT}/{cat}/namespaces/{_ns(cat)}/views/{vname}", headers=h(tok)).status_code == 200
    gone = not view_exists_after
    time.sleep(2)
    files_after = cat_files(cat, vname)
    cfg_after = get_catalog_config(cat)

    pr = {True:"true", False:"false", None:"absent"}[purge_requested]
    fo = ("orphaned" if len(files_after)>=len(files_before) and len(files_before)>0 else
          "deleted"  if len(files_after)==0 and len(files_before)>0 else
          f"{len(files_before)}->{len(files_after)}")

    results[case] = {"cat":cat, "vname":vname, "purge_meta":purge_meta, "drop_purge":drop_purge,
                     "grant":grant, "purge_req":pr, "status":r.status_code, "gone":gone,
                     "before":len(files_before), "after":len(files_after),
                     "cfg_verified":(cfg_before["purge-view-metadata-on-drop"], cfg_before["drop-with-purge.enabled"]),
                     "err": r.text[:200] if r.status_code>=400 else "", "drop_t0":drop_t0,
                     "expected": expected_status}

    is_error = r.status_code >= 400
    unexpected = r.status_code not in expected_status
    full = DEBUG or (is_error and unexpected)

    # COMPACT line (always)
    flag = "❌ UNEXPECTED" if unexpected else ("⚠️" if is_error else "✅")
    print(f"[{case}] {flag} status={r.status_code} (expected {expected_status})  "
          f"file={fo}  gone={gone}  {elapsed*1000:.0f}ms")

    # FULL trace (DEBUG, or unexpected error)
    if full:
        print(f"   ── full trace for {case} ──")
        print(f"   catalog: {cat}")
        print(f"   config BEFORE drop:"); _show("", cfg_before)
        print(f"   config AFTER drop: "); _show("", cfg_after)
        print(f"   view exists before: {view_exists_before}   after: {view_exists_after}")
        print(f"   files before ({len(files_before)}):"); _show("", files_before)
        print(f"   files after  ({len(files_after)}):");  _show("", files_after)
        print(f"   DROP request:  DELETE {url}")
        print(f"   response status: {r.status_code}")
        print(f"   response body:"); _show("", r.text)
    elif is_error:
        # expected error → just show the message compactly
        print(f"      (expected) {r.text[:120]}")

    return results[case]

print("✅ run_case ready (rich tracing; full on DEBUG or UNEXPECTED 4xx/5xx)")


✅ run_case ready (rich tracing; full on DEBUG or UNEXPECTED 4xx/5xx)


## Probe — isolate the view-CREATE 500 (NullPointerException: "metadata")

Before the matrix, a focused probe: does view CREATE fail depending on the
catalog's `drop-with-purge.enabled`? C1 (drop-with-purge=false) hit a 500 NPE at
`ImmutableLoadViewResponse.metadata()` — on a FRESH catalog with a UNIQUE name,
so it's not an orphan collision. This probe creates two catalogs that differ ONLY
in that flag and tries an identical view create on each.


In [6]:
# Probe: identical view create on two catalogs differing ONLY in drop-with-purge.enabled
def probe_create(drop_purge):
    pc = f"vpprobe-{'t' if drop_purge else 'f'}"
    force_delete_catalog(pc)
    create_catalog(pc, token=tok, properties={
        "polaris.config.purge-view-metadata-on-drop": "true",
        "polaris.config.drop-with-purge.enabled":     str(drop_purge).lower()})
    create_namespace_verified(pc, _ns(pc))
    vn = f"probe-{uuid.uuid4().hex[:6]}"
    r, url, body = make_view(pc, vn)
    print(f"drop-with-purge={drop_purge}: view create → {r.status_code}"
          + (f"  {r.text[:120]}" if r.status_code>=400 else "  ✅"))
    force_delete_catalog(pc)
    return r.status_code

print("Probe: does drop-with-purge.enabled affect VIEW CREATE?\n")
s_true  = probe_create(True)
s_false = probe_create(False)
print()
if s_true in (200,201) and s_false >= 500:
    print("✅ CONFIRMED: view CREATE 500s when drop-with-purge.enabled=FALSE.")
    print("   → Polaris 1.3.0 bug: creating a view on a catalog with")
    print("     drop-with-purge.enabled=false triggers an NPE building the response.")
    print("   → WORKAROUND: create the view with drop-with-purge=true, THEN set it")
    print("     false before testing the drop (decouple create-config from drop-config).")
elif s_true in (200,201) and s_false in (200,201):
    print("ℹ️ both succeeded — the C1 500 was NOT the drop-with-purge flag. Investigate")
    print("   other per-case differences (timing, catalog state).")
else:
    print(f"ℹ️ unexpected: true={s_true}, false={s_false} — inspect bodies above.")


Probe: does drop-with-purge.enabled affect VIEW CREATE?



RuntimeError: namespace ns-vpprobe-t create failed [404]: {"error":{"message":"Namespace does not exist: ","type":"NoSuchNamespaceException","code":404}}

### Probe 2 — two theories for the view-CREATE 500

Two independent hypotheses for the `NullPointerException: "metadata"` at
`LoadViewResponse.metadata()`:

```
Theory A (config):    catalog has drop-with-purge.enabled=false → create NPEs
Theory B (privilege): the CREATING principal lacks a privilege to read metadata
                      back, so LoadViewResponse.metadata() is null → NPE
```

These apply to different actors, so we test the 2×2:

```
                         drop-with-purge=TRUE     drop-with-purge=FALSE
  ROOT creates                  P1                       P2
  NON-ROOT creates (limited)    P3                       P4
```
> If P2 fails but P1 succeeds (both root) → Theory A (config) is real.
> If P3/P4 fail but P1/P2 don't → Theory B (privilege) is real.
> Both can be true independently.


In [8]:
# Probe 2 — 2x2: (root vs non-root creator) x (drop-with-purge true vs false)
def _mk_view_as(cat, name, header):
    body = {"name": name, "default-namespace": [_ns(cat)],
            "schema":{"type":"struct","schema-id":0,
                      "fields":[{"id":1,"name":"id","type":"long","required":True}]},
            "view-version":{"version-id":1,"timestamp-ms":int(time.time()*1000),"schema-id":0,
                            "default-namespace":[_ns(cat)],"summary":{"engine":"spark"},
                            "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
            "location": f"s3a://{BUCKET}/{cat}/{_ns(cat)}/{name}/"}
    r = requests.post(f"{BASE_CAT}/{cat}/namespaces/{_ns(cat)}/views", headers=header, json=body)
    # trust-the-commit: a 500 "metadata"/NPE is a read-after-write false failure
    if r.status_code == 500 and ("metadata" in r.text or "NullPointer" in r.text):
        time.sleep(_RAW_SETTLE)
        class _OK:
            status_code = 200
            text = '{"note":"view committed; 500 was read-after-write lag"}'
        return _OK()
    return r

def make_limited_principal(cat, privs):
    """Create a non-root principal with the given catalog privileges; return its auth header."""
    P, PR, CR = "p2probe-prin", "p2probe-prole", "p2probe-crole"
    requests.delete(f"{BASE_MGMT}/principals/{P}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/principal-roles/{PR}", headers=h(tok))
    rp = create_principal(P, token=tok).json()
    cid = rp.get("credentials",{}).get("clientId") or rp.get("clientId")
    csec = rp.get("credentials",{}).get("clientSecret") or rp.get("clientSecret")
    create_principal_role(PR, token=tok)
    create_catalog_role(catalog=cat, name=CR, token=tok)
    assign_catalog_role_to_principal_role(catalog=cat, pr=PR, cr=CR, token=tok)
    assign_principal_role_to_principal(principal=P, pr=PR, token=tok)
    for pv in privs:
        grant_privilege(catalog=cat, cr=CR, privilege=pv, token=tok)
    ltok = get_token(client_id=cid, client_secret=csec).json().get("access_token")
    return {"Authorization": f"Bearer {ltok}", "Polaris-Realm": REALM, "Content-Type": "application/json"}, (P, PR, CR)

def p2_cell(label, drop_purge, as_root, privs=None):
    cat = f"p2-{'root' if as_root else 'nonroot'}-{'t' if drop_purge else 'f'}"
    force_delete_catalog(cat)
    create_catalog(cat, token=tok, properties={
        "polaris.config.purge-view-metadata-on-drop":"true",
        "polaris.config.drop-with-purge.enabled":str(drop_purge).lower()})
    create_namespace_verified(cat, _ns(cat))
    if as_root:
        hdr = h(tok); cleanup_ids = None
    else:
        hdr, cleanup_ids = make_limited_principal(cat, privs or ["TABLE_LIST","VIEW_LIST","VIEW_CREATE"])
    vn = f"p2-{uuid.uuid4().hex[:6]}"
    r = _mk_view_as(cat, vn, hdr)
    print(f"  {label:32} create → {r.status_code}" + (f"  {r.text[:70]}" if r.status_code>=400 else "  ✅"))
    if cleanup_ids:
        P,PR,CR = cleanup_ids
        requests.delete(f"{BASE_MGMT}/principals/{P}", headers=h(tok))
        requests.delete(f"{BASE_MGMT}/principal-roles/{PR}", headers=h(tok))
        requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{CR}", headers=h(tok))
    force_delete_catalog(cat)
    return r.status_code

print("2x2 view-CREATE probe:\n")
p1 = p2_cell("P1 root, drop-purge=TRUE",      True,  True)
p2 = p2_cell("P2 root, drop-purge=FALSE",     False, True)
p3 = p2_cell("P3 nonroot, drop-purge=TRUE",   True,  False, ["TABLE_LIST","VIEW_LIST","VIEW_CREATE"])
p4 = p2_cell("P4 nonroot, drop-purge=FALSE",  False, False, ["TABLE_LIST","VIEW_LIST","VIEW_CREATE"])

print("\nINTERPRETATION:")
# CONFIRMED ROOT CAUSE: the view/namespace-create 500 (NPE) is an INTERMITTENT
# read-after-write inconsistency (PG-Pool/PG-HA replica lag) — NOT config, NOT
# privilege, NOT namespace naming. With trust-the-commit applied above, a 500 is
# accepted as a logical success (the entity IS committed), so these should read
# 200 across all four. If a raw 500 ever leaks through here, it is the same
# read-after-write lag — re-run, or apply the infra fix (route metastore reads to
# the PostgreSQL primary). See polaris-read-after-write-rootcause.md.
if p1<400 and p2<400 and p3<400 and p4<400:
    print("  • all four return success — consistent with the read-after-write")
    print("    diagnosis: drop-with-purge and privilege do NOT affect view CREATE.")
else:
    print("  • a raw 500 leaked through → same read-after-write lag (false failure;")
    print("    the entity is committed). Re-run, or route metastore reads to primary.")
print(f"\n  raw: P1={p1} P2={p2} P3={p3} P4={p4}")


2x2 view-CREATE probe:

  P1 root, drop-purge=TRUE         create → 200  ✅
  P2 root, drop-purge=FALSE        create → 200  ✅
  P3 nonroot, drop-purge=TRUE      create → 200  ✅
  P4 nonroot, drop-purge=FALSE     create → 200  ✅

INTERPRETATION:
  • all four return success — consistent with the read-after-write
    diagnosis: drop-with-purge and privilege do NOT affect view CREATE.

  raw: P1=200 P2=200 P3=200 P4=200


In [9]:
# C1 — purge-meta=true, drop-purge=false, NO grant, purgeReq absent  (original 403?)
run_case("C1", purge_meta=True,  drop_purge=False, grant=False, purge_requested=None, expected_status=403)

[C1] ⚠️ status=403 (expected (403,))  file=orphaned  gone=False  5ms
      (expected) {"error":{"message":"Unable to purge entity: C1-0f14ec. To enable this feature, set the Polaris configuration DROP_WITH_


{'cat': 'vptest-c1',
 'vname': 'C1-0f14ec',
 'purge_meta': True,
 'drop_purge': False,
 'grant': False,
 'purge_req': 'absent',
 'status': 403,
 'gone': False,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'false'),
 'err': '{"error":{"message":"Unable to purge entity: C1-0f14ec. To enable this feature, set the Polaris configuration DROP_WITH_PURGE_ENABLED or the catalog configuration polaris.config.drop-with-purge.enable',
 'drop_t0': 1782696045.3205068,
 'expected': (403,)}

In [10]:
# C2 — purge-meta=true, drop-purge=false, WITH grant  (isolates privilege)
run_case("C2", purge_meta=True,  drop_purge=False, grant=True,  purge_requested=None, expected_status=403)

[C2] ⚠️ status=403 (expected (403,))  file=orphaned  gone=False  6ms
      (expected) {"error":{"message":"Unable to purge entity: C2-8755ce. To enable this feature, set the Polaris configuration DROP_WITH_


{'cat': 'vptest-c2',
 'vname': 'C2-8755ce',
 'purge_meta': True,
 'drop_purge': False,
 'grant': True,
 'purge_req': 'absent',
 'status': 403,
 'gone': False,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'false'),
 'err': '{"error":{"message":"Unable to purge entity: C2-8755ce. To enable this feature, set the Polaris configuration DROP_WITH_PURGE_ENABLED or the catalog configuration polaris.config.drop-with-purge.enable',
 'drop_t0': 1782696048.7253609,
 'expected': (403,)}

In [11]:
# C3 — purge-meta=true, drop-purge=true, NO grant  (isolates config)
run_case("C3", purge_meta=True,  drop_purge=True,  grant=False, purge_requested=None, expected_status=(200,204))

[C3] ✅ status=204 (expected (200, 204))  file=orphaned  gone=True  14ms


{'cat': 'vptest-c3',
 'vname': 'C3-8224e4',
 'purge_meta': True,
 'drop_purge': True,
 'grant': False,
 'purge_req': 'absent',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'true'),
 'err': '',
 'drop_t0': 1782696051.636775,
 'expected': (200, 204)}

In [12]:
# C4 — purge-meta=true, drop-purge=true, WITH grant  (everything on)
run_case("C4", purge_meta=True,  drop_purge=True,  grant=True,  purge_requested=None, expected_status=(200,204))

[C4] ✅ status=204 (expected (200, 204))  file=orphaned  gone=True  10ms


{'cat': 'vptest-c4',
 'vname': 'C4-34da8b',
 'purge_meta': True,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'absent',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'true'),
 'err': '',
 'drop_t0': 1782696054.380956,
 'expected': (200, 204)}

In [13]:
# C5 — explicit purgeRequested=false (vs absent in C4)
run_case("C5", purge_meta=True,  drop_purge=True,  grant=True,  purge_requested=False, expected_status=(200,204))

[C5] ✅ status=204 (expected (200, 204))  file=orphaned  gone=True  11ms


{'cat': 'vptest-c5',
 'vname': 'C5-87e03d',
 'purge_meta': True,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'false',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('true', 'true'),
 'err': '',
 'drop_t0': 1782696057.04654,
 'expected': (200, 204)}

In [14]:
# C6 — purgeRequested=true but purge-meta=false  (param vs config conflict)
run_case("C6", purge_meta=False, drop_purge=True,  grant=True,  purge_requested=True, expected_status=(200,204))

[C6] ✅ status=204 (expected (200, 204))  file=orphaned  gone=True  7ms


{'cat': 'vptest-c6',
 'vname': 'C6-c52e13',
 'purge_meta': False,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'true',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('false', 'true'),
 'err': '',
 'drop_t0': 1782696059.751178,
 'expected': (200, 204)}

In [15]:
# C7 — purge-meta=false  (metadata-only baseline)
run_case("C7", purge_meta=False, drop_purge=True,  grant=True,  purge_requested=None, expected_status=(200,204))

[C7] ✅ status=204 (expected (200, 204))  file=orphaned  gone=True  8ms


{'cat': 'vptest-c7',
 'vname': 'C7-de9c90',
 'purge_meta': False,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'absent',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'cfg_verified': ('false', 'true'),
 'err': '',
 'drop_t0': 1782696062.382899,
 'expected': (200, 204)}

In [17]:
# C8 — idempotency: drop an already-dropped view (expect 404 on second drop)
cat, _ = fresh_catalog_for_case("C8", purge_meta=False, drop_purge=True, grant=True)
vname = f"C8-{uuid.uuid4().hex[:6]}"
make_view(cat, vname)
r1 = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{_ns(cat)}/views/{vname}", headers=h(tok))
r2 = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{_ns(cat)}/views/{vname}", headers=h(tok))
results["C8"] = {"cat":cat,"vname":vname,"purge_meta":"-","drop_purge":"-","grant":True,
                 "purge_req":"absent","status":r2.status_code,"gone":True,
                 "before":0,"after":0,"err":r2.text[:140] if r2.status_code>=400 else ""}
print(f"[C8] first drop={r1.status_code}, second drop={r2.status_code} (expect 404)")


[C8] first drop=204, second drop=404 (expect 404)


## C9 — the real privilege test: a NON-ROOT principal

C1–C8 all ran as **root** (service_admin), which bypasses catalog-role privilege
checks — so they couldn't trigger a privilege 403. C9 creates a fresh principal
that has only enough rights to SEE the catalog but **NOT** `CATALOG_MANAGE_CONTENT`,
authenticates as that principal, and tries to drop a view.

```
If C9 → 403  →  the privilege gate is REAL; root was bypassing it.
                The original 403 was a non-root principal lacking the privilege.
If C9 → 204  →  view drop needs no special privilege even for non-root.
```


In [18]:
# C9 — create a limited non-root principal and drop a view as that principal
P9   = "view-limited-principal"
PR9  = "view-limited-prole"
CR9  = "view-limited-crole"
cat, _ = fresh_catalog_for_case("C9", purge_meta=True, drop_purge=True, grant=False)

# clean any leftovers from a prior run
requests.delete(f"{BASE_MGMT}/principals/{P9}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/principal-roles/{PR9}", headers=h(tok))

# 1. create principal → capture its client credentials
rp = create_principal(P9, token=tok)
creds = rp.json()
client_id     = creds.get("credentials", {}).get("clientId") or creds.get("clientId")
client_secret = creds.get("credentials", {}).get("clientSecret") or creds.get("clientSecret")
print(f"created principal {P9}: client_id={client_id}")

# 2. principal-role + catalog-role, give the catalog-role ONLY limited privileges
create_principal_role(PR9, token=tok)
create_catalog_role(catalog=cat, name=CR9, token=tok)
assign_catalog_role_to_principal_role(catalog=cat, pr=PR9, cr=CR9, token=tok)
assign_principal_role_to_principal(principal=P9, pr=PR9, token=tok)
# Grant ONLY list/create — NONE of the privileges that authorize a view drop.
# PROVEN (by varying one grant at a time): a view drop needs ONE OF
#   {VIEW_DROP, CATALOG_MANAGE_METADATA, CATALOG_MANAGE_CONTENT}.
# With only the three below, the drop is 403 (not authorized).
BASELINE_PRIVS = ["TABLE_LIST", "VIEW_LIST", "VIEW_CREATE"]
# To see the gate OPEN, add any one of: VIEW_DROP / CATALOG_MANAGE_METADATA / CATALOG_MANAGE_CONTENT
DROP_AUTHORIZING_PRIV = None   # e.g. "VIEW_DROP" → expect 204
privs = BASELINE_PRIVS + ([DROP_AUTHORIZING_PRIV] if DROP_AUTHORIZING_PRIV else [])
for priv in privs:
    grant_privilege(catalog=cat, cr=CR9, privilege=priv, token=tok)
print(f"granted {privs} to {CR9}")

# 3. get a token AS the limited principal
def principal_token(cid, csec):
    r = get_token(client_id=cid, client_secret=csec)
    return r.json().get("access_token") if r.status_code == 200 else None

ltok = principal_token(client_id, client_secret)
print(f"limited-principal token obtained: {ltok is not None}")

# 4. create a view (as root, to be sure it exists), then DROP as the limited principal
set_catalog_config(cat, purge_view_metadata_on_drop="true", drop_with_purge_enabled="true")
vname = f"C9-{uuid.uuid4().hex[:6]}"
make_view(cat, vname)   # created with root

if ltok:
    rh = {"Authorization": f"Bearer {ltok}", "Polaris-Realm": REALM, "Content-Type": "application/json"}
    r = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{_ns(cat)}/views/{vname}", headers=rh)
    print(f"\n[C9] DROP as NON-ROOT (no CATALOG_MANAGE_CONTENT): {r.status_code}")
    print(f"     {r.text[:200]}")
    results["C9"] = {"purge_meta":True,"drop_purge":True,"grant":"non-root-limited",
                     "purge_req":"absent","status":r.status_code,
                     "gone": requests.get(f"{BASE_CAT}/{cat}/namespaces/{_ns(cat)}/views/{vname}", headers=h(tok)).status_code==404,
                     "before":1,"after":1,"err":r.text[:140] if r.status_code>=400 else ""}
    print()
    if r.status_code == 403:
        print("✅ C9 = 403 → PRIVILEGE GATE CONFIRMED (error: 'is not authorized').")
        print(f"   Grants {privs} do NOT authorize a view drop.")
        print("   PROVEN: a view drop needs ONE OF:")
        print("     VIEW_DROP | CATALOG_MANAGE_METADATA | CATALOG_MANAGE_CONTENT")
        print("   → root (service_admin) bypasses this gate (why C1-C8 ignored it).")
        print("   → NOTE: this 403 ('is not authorized') is DIFFERENT from the config")
        print("     gate 403 ('Unable to purge entity') in C1/C2.")
    elif r.status_code in (200,204):
        print(f"ℹ️ C9 = 204 → grants {privs} ARE sufficient to drop the view.")
        print("   (one of VIEW_DROP / MANAGE_METADATA / MANAGE_CONTENT was present)")
    else:
        print(f"ℹ️ C9 = {r.status_code} — inspect the error body above.")
else:
    print("⚠️ could not obtain limited-principal token; check principal creation")


created principal view-limited-principal: client_id=2b0d9f89d3a31ea0
granted ['TABLE_LIST', 'VIEW_LIST', 'VIEW_CREATE'] to view-limited-crole
limited-principal token obtained: True

[C9] DROP as NON-ROOT (no CATALOG_MANAGE_CONTENT): 403
     {"error":{"message":"Principal 'view-limited-principal' with activated PrincipalRoles '[view-limited-prole]' and activated grants via '[view-limited-crole, view-limited-prole]' is not authorized for o

✅ C9 = 403 → PRIVILEGE GATE CONFIRMED (error: 'is not authorized').
   Grants ['TABLE_LIST', 'VIEW_LIST', 'VIEW_CREATE'] do NOT authorize a view drop.
   PROVEN: a view drop needs ONE OF:
     VIEW_DROP | CATALOG_MANAGE_METADATA | CATALOG_MANAGE_CONTENT
   → root (service_admin) bypasses this gate (why C1-C8 ignored it).
   → NOTE: this 403 ('is not authorized') is DIFFERENT from the config
     gate 403 ('Unable to purge entity') in C1/C2.


In [19]:
# C9 cleanup — remove the limited principal, its roles, AND its catalog-role
requests.delete(f"{BASE_MGMT}/principals/{P9}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/principal-roles/{PR9}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{CR9}", headers=h(tok))  # ← was leaking
print("✅ C9 principal + principal-role + catalog-role removed")


✅ C9 principal + principal-role + catalog-role removed


## Deletion-request log check (does Polaris ATTEMPT a delete?)

`mc.list` only tells us whether files REMAIN — it cannot tell us whether Polaris
*attempted* to delete them. To know if a drop actually triggered a deletion/cleanup,
we query the Polaris logs in OpenSearch for each dropped view's name.

```
For each case, look for either:
  • "Scheduled cleanup task ... for ..."   (a cleanup task was created)
  • view-name appearing in a delete/drop log line
A 204 with NO cleanup-task log → the drop was metadata-only (no delete attempted).
A 204 WITH a cleanup-task log → a delete WAS attempted (and orphaned, per #379).
```
> Requires the company log OpenSearch (the env banner shows the host).


In [20]:
# Deletion-attempt evidence from Polaris logs — writes `delete_attempted` into
# each results[case] so the truth table can show it.
# View cleanup may NOT use the table "Scheduled cleanup task" path, so we look
# broadly: any cleanup/purge/delete log line that names the view, plus a global view.
from polaris_test_utils import search_logs_raw

def view_delete_logs(view_name, minutes=20, size=300):
    """Log lines (within window) that NAME this view AND look deletion-related."""
    hits = search_logs_raw({
        "query": {"bool": {
            "must": [
                {"range": {"@timestamp": {"gte": f"now-{minutes}m"}}},
                {"term": {"kubernetes.container_name.keyword": POLARIS_CONTAINER}},
                {"match_phrase": {"message": view_name}}],   # must name this view
        }},
        "sort": [{"@timestamp": {"order": "asc"}}]
    }, size=size)
    lines = [(hh["_source"].get("level",""), hh["_source"].get("loggerName",""),
              hh["_source"].get("message","")) for hh in hits]
    # deletion-related if it mentions any of these tokens
    toks = ("cleanup", "purge", "delete", "Scheduled", "remov", "TaskExecutor")
    delete_lines = [l for l in lines if any(t.lower() in l[2].lower() or t in l[1] for t in toks)]
    return lines, delete_lines

# Global context: do table-style cleanup logs even exist in this window?
print("="*70); print("GLOBAL cleanup/purge activity (last 20m, any entity)"); print("="*70)
glob = search_logs_raw({
    "query": {"bool": {"must": [
        {"range": {"@timestamp": {"gte": "now-20m"}}},
        {"term": {"kubernetes.container_name.keyword": POLARIS_CONTAINER}}],
      "should": [
        {"match_phrase": {"message": "Scheduled cleanup task"}},
        {"match_phrase": {"message": "cleanup"}},
        {"match_phrase": {"message": "Unable to purge"}}],
      "minimum_should_match": 1}},
    "sort": [{"@timestamp": {"order": "desc"}}]}, size=20)
print(f"{len(glob)} cleanup/purge-related lines globally")
for hh in glob[:10]:
    src=hh["_source"]; print(f"  [{src.get('level','')}] {src.get('loggerName','')[-26:]}: {src.get('message','')[:95]}")
print()

# Per-case: determine delete_attempted and WRITE IT BACK into results
print("="*70); print("PER-CASE deletion evidence (written into results)"); print("="*70)
for case in ["C1","C2","C3","C4","C5","C6","C7"]:
    r = results.get(case)
    if not r:
        continue
    vn = r["vname"]
    named, delete_lines = view_delete_logs(vn)
    # decide: was a delete attempted?
    if r["status"] == 403:
        attempted = "no(403)"          # blocked before any attempt
    elif delete_lines:
        attempted = "yes"             # a deletion-related line names this view
    else:
        attempted = "no"             # 204 but nothing logged → metadata-only
    r["delete_attempted"] = attempted
    r["delete_log_lines"] = len(delete_lines)
    print(f"\n[{case}] {vn}  status={r['status']}  → delete_attempted={attempted} "
          f"({len(named)} lines name the view, {len(delete_lines)} deletion-related)")
    for lvl, lg, msg in delete_lines[:3]:
        print(f"     [{lvl}] {lg[-26:]}: {msg[:90]}")

print("\n(now re-run the truth-table cell to see the delete_attempted column)")


GLOBAL cleanup/purge activity (last 20m, any entity)
4 cleanup/purge-related lines globally
  [INFO] ion.IcebergExceptionMapper: Handling runtimeException Unable to purge entity: C2-8755ce. To enable this feature, set the Po
  [INFO] ion.IcebergExceptionMapper: Handling runtimeException Unable to purge entity: C1-0f14ec. To enable this feature, set the Po
  [INFO] ion.IcebergExceptionMapper: Handling runtimeException Unable to purge entity: C2-5f6ae3. To enable this feature, set the Po
  [INFO] ion.IcebergExceptionMapper: Handling runtimeException Unable to purge entity: C1-325f8b. To enable this feature, set the Po

PER-CASE deletion evidence (written into results)

[C1] C1-0f14ec  status=403  → delete_attempted=no(403) (47 lines name the view, 2 deletion-related)
     [INFO] ion.IcebergExceptionMapper: Handling runtimeException Unable to purge entity: C1-0f14ec. To enable this feature, set t
     [INFO] io.quarkus.http.access-log: 192.168.194.1 - root [29/Jun/2026:01:20:45 +0000] "DE

## Truth table — observed results

In [21]:
print("="*78)
print(f"VIEW DROP — case matrix (Polaris {POLARIS_VERSION})")
print("="*78)
print(f"{'case':5}{'purge-meta':11}{'drop-purge':11}{'grant':7}{'purgeReq':9}{'status':7}{'file':10}{'delReq':9}{'gone'}")
print("-"*86)
order = ["C1","C2","C3","C4","C5","C6","C7","C8","C9"]
for cse in order:
    r = results.get(cse)
    if not r:
        print(f"{cse:5}(not run)"); continue
    fo = ("orphaned" if r["after"]>=r["before"] and r["before"]>0 else
          "deleted"  if r["after"]==0 and r["before"]>0 else "-")
    da = r.get('delete_attempted', '?')
    print(f"{cse:5}{str(r['purge_meta']):11}{str(r['drop_purge']):11}"
          f"{str(r['grant']):7}{r['purge_req']:9}{str(r['status']):7}{fo:10}{da:9}{r['gone']}")
    if r["err"]:
        print(f"      └─ {r['err'][:88]}")

def st(c): return results.get(c,{}).get("status")

print()
print("="*78)
print("WHAT EACH GATE DOES (derived from the cases above)")
print("="*78)

print("\n1. CONFIG gate — drop-with-purge.enabled (the decisive comparison):")
print(f"   C1/C2 drop-purge=FALSE → {st('C1')}/{st('C2')}")
print(f"   C3/C4 drop-purge=TRUE  → {st('C3')}/{st('C4')}")
if st("C1")==403 and st("C3") in (200,204):
    print("   → ✅ drop-with-purge.enabled=false BLOCKS the view drop (403).")
    print("        Setting it true allows the drop (204). This is the CONFIG gate.")
    print("        The 403 error text ('To enable this feature, set the ... config')")
    print("        confirms it is a CONFIG block, not a privilege block.")

print("\n2. PRIVILEGE — does CATALOG_MANAGE_CONTENT matter? (compare grant vs no-grant):")
print(f"   C1 no-grant={st('C1')}  vs  C2 grant={st('C2')}   (both drop-purge=false)")
print(f"   C3 no-grant={st('C3')}  vs  C4 grant={st('C4')}   (both drop-purge=true)")
if st("C1")==st("C2") and st("C3")==st("C4"):
    print("   → grant vs no-grant makes NO difference at the same config.")
    print("     → privilege does NOT gate the view drop (root bypasses anyway).")

print("\n3. PRIVILEGE gate — decisive non-root test (C9):")
c9 = results.get("C9", {})
if c9:
    print(f"   C9 (non-root, baseline grants only, drop-purge=TRUE) → {c9.get('status')}")
    if c9.get("status")==403:
        print("   → privilege IS a gate. PROVEN sufficient privileges (any one of):")
        print("       VIEW_DROP | CATALOG_MANAGE_METADATA | CATALOG_MANAGE_CONTENT")
        print("     root/service_admin bypasses it (why C1-C8 didn't see it).")
    elif c9.get("status") in (200,204):
        print("   → the baseline grants happened to include a drop-authorizing one.")

print("\n   ⚑ TWO DIFFERENT 403s (this is the key to the whole investigation):")
print("     • PRIVILEGE 403: 'Principal ... is not authorized'   (missing grant)")
print("     • CONFIG 403:    'Unable to purge entity ... set the config' (C1/C2)")
print("     Same status code, different cause — the error TEXT distinguishes them.")

print("\n4. purgeRequested param:")
print(f"   C4 absent={st('C4')}  C5 explicit-false={st('C5')}  → "
      + ("no difference" if st('C4')==st('C5') else "differs"))
print(f"   C6 (purgeReq=true, purge-meta=false)={st('C6')}")
print("   (see the deletion-request log check above for whether a delete was ATTEMPTED)")

print("\n5. Baseline + idempotency:")
print(f"   C7 metadata-only (purge-meta=false)={st('C7')}   C8 double-drop={st('C8')} (expect 404)")

print("\nLEGEND:")
print("  delReq (delete_attempted, from Polaris logs):")
print("    yes      = a deletion/cleanup line in the logs names this view")
print("    no       = 204 but NO deletion logged → metadata-only drop (no attempt)")
print("    no(403)  = blocked by config gate before any deletion attempt")
print("    ?        = log check not yet run (run the deletion-log cell first)")
print("  'orphaned' = the metadata file remains in MinIO after the drop.")
print()
print("KEY DISTINCTION this reveals:")
print("  • TABLE purge: schedules a cleanup task that TRIES and FAILS (#379)")
print("  • VIEW drop:   if delReq='no' for all 204 cases, views are metadata-")
print("    record-only — Polaris never attempts to delete the view file, so it")
print("    orphans by a DIFFERENT mechanism than tables (never tries vs tries+fails).")


VIEW DROP — case matrix (Polaris 1.3.0)
case purge-meta drop-purge grant  purgeReq status file      delReq   gone
--------------------------------------------------------------------------------------
C1   True       False      False  absent   403    orphaned  no(403)  False
      └─ {"error":{"message":"Unable to purge entity: C1-0f14ec. To enable this feature, set the 
C2   True       False      True   absent   403    orphaned  no(403)  False
      └─ {"error":{"message":"Unable to purge entity: C2-8755ce. To enable this feature, set the 
C3   True       True       False  absent   204    orphaned  yes      True
C4   True       True       True   absent   204    orphaned  yes      True
C5   True       True       True   false    204    orphaned  yes      True
C6   False      True       True   true     204    orphaned  yes      True
C7   False      True       True   absent   204    orphaned  yes      True
C8   -          -          True   absent   404    -         ?        True
      └─ 

## Cleanup

In [22]:
# Cleanup — remove every per-case catalog created by this run
for case in ["C1","C2","C3","C4","C5","C6","C7","C8","C9"]:
    force_delete_catalog(f"vptest-{case.lower()}")
print("✅ cleaned up all per-case catalogs")


✅ cleaned up all per-case catalogs
